# CNN From Scratch — Inference Notebook (CIFAR-10)

**Purpose:** Train ပြီးသား CNN model ကို load လုပ်ပြီး input image တစ်ခုကို predict လုပ်ပေးမယ့် notebook

**Features:**
- Input image ကို preprocess လုပ်ပြီး model ထဲထည့်
- Raw logits ထုတ်ပြ
- Softmax probabilities ထုတ်ပြ
- Top-K predictions bar chart
- GradCAM visualization (model ဘယ်နေရာကိုကြည့်ပြီး predict လုပ်လဲ)
- Confidence gauge + decision summary

In [ ]:
# --- 1. Imports ---
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import transforms
from PIL import Image
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import numpy as np
import os

In [ ]:
# --- 2. Device & Config ---
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# CIFAR-10 class names
CLASS_NAMES = ['airplane', 'automobile', 'bird', 'cat', 'deer',
               'dog', 'frog', 'horse', 'ship', 'truck']

NUM_CLASSES = len(CLASS_NAMES)

# CIFAR-10 normalization stats
CIFAR10_MEAN = (0.4914, 0.4822, 0.4465)
CIFAR10_STD  = (0.2470, 0.2435, 0.2616)

# Path to saved model weights
MODEL_PATH = "best_cnn_cifar10.pth"

In [ ]:
# --- 3. Model Architecture (must match training) ---
class CNN(nn.Module):
    def __init__(self, num_classes=10):
        super(CNN, self).__init__()

        # Block 1: 3 → 32 channels
        self.block1 = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.Conv2d(32, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Dropout2d(0.25)
        )

        # Block 2: 32 → 64 channels
        self.block2 = nn.Sequential(
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.Conv2d(64, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Dropout2d(0.25)
        )

        # Block 3: 64 → 128 channels
        self.block3 = nn.Sequential(
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.Conv2d(128, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Dropout2d(0.25)
        )

        # Classifier
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128 * 4 * 4, 512),
            nn.BatchNorm1d(512),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(512, num_classes)
        )

    def forward(self, x):
        x = self.block1(x)
        x = self.block2(x)
        x = self.block3(x)
        x = self.classifier(x)
        return x

print("✅ Model architecture defined.")

In [ ]:
# --- 4. Load Trained Weights ---
model = CNN(num_classes=NUM_CLASSES).to(device)
model.load_state_dict(torch.load(MODEL_PATH, map_location=device, weights_only=True))
model.eval()
print(f"✅ Model loaded from '{MODEL_PATH}' — set to eval mode.")

In [ ]:
# --- 5. Image Preprocessing Pipeline ---
inference_transform = transforms.Compose([
    transforms.Resize((32, 32)),          # CIFAR-10 input size
    transforms.ToTensor(),
    transforms.Normalize(CIFAR10_MEAN, CIFAR10_STD)
])

def load_and_preprocess(image_path: str) -> tuple:
    """Load image, return (original_pil, preprocessed_tensor)"""
    img_pil = Image.open(image_path).convert("RGB")
    img_tensor = inference_transform(img_pil).unsqueeze(0)  # (1, 3, 32, 32)
    return img_pil, img_tensor.to(device)

print("✅ Preprocessing pipeline ready.")

## 🔍 Single Image Inference

Input image path ထည့်ပြီး run ပါ။ Model က:
1. **Raw logits** (unnormalized scores) ထုတ်ပြပေးမယ်
2. **Softmax probabilities** (0–1 confidence) ထုတ်ပြပေးမယ်
3. **Predicted label** ဘာပုံဖြစ်တယ်ဆိုတာ ပြပေးမယ်

In [ ]:
# --- 6. Inference Function ---
def predict(model, image_path: str):
    """Run inference on a single image and return all results."""
    img_pil, img_tensor = load_and_preprocess(image_path)

    with torch.no_grad():
        logits = model(img_tensor)                          # raw logits  (1, 10)
        probabilities = F.softmax(logits, dim=1)            # softmax     (1, 10)
        confidence, predicted_idx = torch.max(probabilities, dim=1)

    logits_np = logits.cpu().squeeze().numpy()
    probs_np  = probabilities.cpu().squeeze().numpy()
    pred_class = CLASS_NAMES[predicted_idx.item()]
    conf_value = confidence.item()

    return {
        "image_pil": img_pil,
        "logits": logits_np,
        "probabilities": probs_np,
        "predicted_class": pred_class,
        "predicted_index": predicted_idx.item(),
        "confidence": conf_value,
    }

print("✅ predict() function ready.")

In [ ]:
# --- 7. Run Inference ---
# ⬇️ ဒီမှာ image path ထည့်ပါ
IMAGE_PATH = "test_image.jpg"   # <-- သင့် image path ထည့်ပါ

result = predict(model, IMAGE_PATH)

print(f"🏷️  Predicted Class : {result['predicted_class']}")
print(f"📊 Confidence      : {result['confidence']:.4f} ({result['confidence']*100:.2f}%)")
print()
print("=" * 55)
print(f"{'Class':<15} {'Logit':>10} {'Probability':>12}")
print("=" * 55)
for i, cls in enumerate(CLASS_NAMES):
    marker = " ◀" if i == result['predicted_index'] else ""
    print(f"{cls:<15} {result['logits'][i]:>10.4f} {result['probabilities'][i]:>11.4f}{marker}")
print("=" * 55)

## 📊 Visualization — Logits, Probabilities & Confidence

In [ ]:
# --- 8. Full Visualization Dashboard ---
def visualize_prediction(result):
    """Complete visualization: image + logits + probabilities + confidence gauge."""
    fig = plt.figure(figsize=(18, 10))
    gs = gridspec.GridSpec(2, 3, figure=fig, hspace=0.4, wspace=0.35)

    pred_cls = result['predicted_class']
    conf = result['confidence']
    logits = result['logits']
    probs = result['probabilities']

    # --- (A) Original Image ---
    ax_img = fig.add_subplot(gs[0, 0])
    ax_img.imshow(result['image_pil'])
    ax_img.set_title(f"Input Image\nPrediction: {pred_cls} ({conf*100:.1f}%)",
                     fontsize=12, fontweight='bold')
    ax_img.axis('off')

    # --- (B) Raw Logits Bar Chart ---
    ax_logit = fig.add_subplot(gs[0, 1])
    colors_logit = ['#e74c3c' if v < 0 else '#2ecc71' for v in logits]
    colors_logit[result['predicted_index']] = '#3498db'
    bars_l = ax_logit.barh(CLASS_NAMES, logits, color=colors_logit, edgecolor='white')
    ax_logit.set_xlabel('Logit Value')
    ax_logit.set_title('Raw Logits (Before Softmax)', fontsize=11, fontweight='bold')
    ax_logit.axvline(x=0, color='gray', linewidth=0.8, linestyle='--')
    for bar, val in zip(bars_l, logits):
        ax_logit.text(val + 0.1 if val >= 0 else val - 0.1, bar.get_y() + bar.get_height()/2,
                      f'{val:.2f}', va='center', ha='left' if val >= 0 else 'right', fontsize=8)

    # --- (C) Softmax Probabilities Bar Chart ---
    ax_prob = fig.add_subplot(gs[0, 2])
    sorted_indices = np.argsort(probs)[::-1]
    sorted_names = [CLASS_NAMES[i] for i in sorted_indices]
    sorted_probs = probs[sorted_indices]
    colors_prob = ['#3498db' if i == 0 else '#bdc3c7' for i in range(len(sorted_probs))]
    bars_p = ax_prob.barh(sorted_names[::-1], sorted_probs[::-1],
                          color=colors_prob[::-1], edgecolor='white')
    ax_prob.set_xlabel('Probability')
    ax_prob.set_title('Softmax Probabilities (Ranked)', fontsize=11, fontweight='bold')
    ax_prob.set_xlim(0, 1.0)
    for bar, val in zip(bars_p, sorted_probs[::-1]):
        ax_prob.text(val + 0.01, bar.get_y() + bar.get_height()/2,
                     f'{val:.4f}', va='center', fontsize=8)

    # --- (D) Confidence Gauge (pie-style) ---
    ax_gauge = fig.add_subplot(gs[1, 0])
    gauge_colors = ['#2ecc71' if conf >= 0.7 else '#f39c12' if conf >= 0.4 else '#e74c3c', '#ecf0f1']
    ax_gauge.pie([conf, 1 - conf], colors=gauge_colors,
                 startangle=90, counterclock=False,
                 wedgeprops=dict(width=0.35, edgecolor='white'))
    ax_gauge.text(0, 0, f'{conf*100:.1f}%', ha='center', va='center',
                  fontsize=22, fontweight='bold',
                  color=gauge_colors[0])
    ax_gauge.set_title('Confidence Level', fontsize=11, fontweight='bold')

    # --- (E) Top-5 Predictions Table ---
    ax_table = fig.add_subplot(gs[1, 1])
    ax_table.axis('off')
    top5_idx = sorted_indices[:5]
    table_data = []
    for rank, idx in enumerate(top5_idx, 1):
        table_data.append([f"#{rank}", CLASS_NAMES[idx],
                           f"{logits[idx]:.3f}", f"{probs[idx]*100:.2f}%"])
    table = ax_table.table(cellText=table_data,
                           colLabels=['Rank', 'Class', 'Logit', 'Prob'],
                           loc='center', cellLoc='center')
    table.auto_set_font_size(False)
    table.set_fontsize(10)
    table.scale(1, 1.6)
    # Color the top prediction row
    for j in range(4):
        table[1, j].set_facecolor('#d5f4e6')
    ax_table.set_title('Top-5 Predictions', fontsize=11, fontweight='bold')

    # --- (F) Probability Distribution (Pie Chart) ---
    ax_pie = fig.add_subplot(gs[1, 2])
    # Show top-5 + "others"
    top5_probs = probs[sorted_indices[:5]]
    top5_names = [CLASS_NAMES[i] for i in sorted_indices[:5]]
    other_prob = 1.0 - top5_probs.sum()
    if other_prob > 0.001:
        pie_probs = list(top5_probs) + [other_prob]
        pie_names = top5_names + ['others']
    else:
        pie_probs = list(top5_probs)
        pie_names = top5_names
    explode = [0.05] + [0] * (len(pie_probs) - 1)
    ax_pie.pie(pie_probs, labels=pie_names, autopct='%1.1f%%',
               startangle=140, explode=explode, textprops={'fontsize': 9})
    ax_pie.set_title('Probability Distribution', fontsize=11, fontweight='bold')

    plt.suptitle(f"CNN Inference Result — \"{pred_cls}\"",
                 fontsize=15, fontweight='bold', y=1.01)
    plt.tight_layout()
    plt.show()

visualize_prediction(result)

## 🔥 GradCAM Visualization

Model က image ရဲ့ ဘယ်နေရာကို အဓိကကြည့်ပြီး prediction လုပ်လဲဆိုတာ heatmap နဲ့ ပြပေးမယ်။

In [ ]:
# --- 9. GradCAM Implementation ---
class GradCAM:
    """Simple GradCAM for CNN models."""
    def __init__(self, model, target_layer):
        self.model = model
        self.target_layer = target_layer
        self.gradients = None
        self.activations = None

        # Register hooks
        target_layer.register_forward_hook(self._forward_hook)
        target_layer.register_full_backward_hook(self._backward_hook)

    def _forward_hook(self, module, input, output):
        self.activations = output.detach()

    def _backward_hook(self, module, grad_input, grad_output):
        self.gradients = grad_output[0].detach()

    def generate(self, input_tensor, target_class=None):
        self.model.eval()
        output = self.model(input_tensor)

        if target_class is None:
            target_class = output.argmax(dim=1).item()

        self.model.zero_grad()
        target_score = output[0, target_class]
        target_score.backward()

        # Global average pool the gradients → channel weights
        weights = self.gradients.mean(dim=[2, 3], keepdim=True)  # (1, C, 1, 1)
        cam = (weights * self.activations).sum(dim=1, keepdim=True)  # (1, 1, H, W)
        cam = F.relu(cam)
        cam = cam.squeeze().cpu().numpy()

        # Normalize to [0, 1]
        cam = (cam - cam.min()) / (cam.max() - cam.min() + 1e-8)
        return cam, target_class

print("✅ GradCAM class ready.")

In [ ]:
# --- 10. GradCAM Visualization ---
def visualize_gradcam(model, image_path):
    """Generate and display GradCAM heatmap overlay."""
    from matplotlib.colors import Normalize as MplNormalize
    import matplotlib.cm as cm

    # Target last conv block (block3) for GradCAM
    gradcam = GradCAM(model, model.block3)

    img_pil, img_tensor = load_and_preprocess(image_path)
    img_tensor.requires_grad_(True)

    cam, pred_class_idx = gradcam.generate(img_tensor)
    pred_class = CLASS_NAMES[pred_class_idx]

    # Resize CAM to original image size
    img_resized = img_pil.resize((32, 32))
    img_np = np.array(img_resized).astype(np.float32) / 255.0

    # Resize CAM heatmap to image size
    from PIL import Image as PILImage
    cam_resized = np.array(PILImage.fromarray((cam * 255).astype(np.uint8)).resize(
        (32, 32), PILImage.BILINEAR)) / 255.0

    # Apply colormap
    heatmap = cm.jet(cam_resized)[:, :, :3]
    overlay = 0.5 * img_np + 0.5 * heatmap

    # Plot
    fig, axes = plt.subplots(1, 4, figsize=(16, 4))

    axes[0].imshow(img_pil)
    axes[0].set_title("Original Image", fontsize=11)
    axes[0].axis('off')

    axes[1].imshow(img_np)
    axes[1].set_title("Resized (32×32)", fontsize=11)
    axes[1].axis('off')

    axes[2].imshow(cam_resized, cmap='jet')
    axes[2].set_title("GradCAM Heatmap", fontsize=11)
    axes[2].axis('off')

    axes[3].imshow(np.clip(overlay, 0, 1))
    axes[3].set_title(f"Overlay → {pred_class}", fontsize=11, fontweight='bold')
    axes[3].axis('off')

    plt.suptitle(f"GradCAM: Model focuses on these regions for \"{pred_class}\"",
                 fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.show()

visualize_gradcam(model, IMAGE_PATH)

## 📁 Batch Inference (Multiple Images)

Folder ထဲက images တွေကို batch inference run ပြီး results summary ထုတ်ပေးမယ်။

In [ ]:
# --- 11. Batch Inference on a Folder ---
def batch_inference(model, image_folder: str, max_images: int = 20):
    """Run inference on all images in a folder and show grid."""
    valid_exts = {'.jpg', '.jpeg', '.png', '.bmp', '.webp'}
    image_files = [f for f in os.listdir(image_folder)
                   if os.path.splitext(f)[1].lower() in valid_exts][:max_images]

    if not image_files:
        print(f"❌ No images found in '{image_folder}'")
        return

    n = len(image_files)
    cols = min(5, n)
    rows = (n + cols - 1) // cols

    fig, axes = plt.subplots(rows, cols, figsize=(4 * cols, 4 * rows))
    if rows == 1 and cols == 1:
        axes = np.array([axes])
    axes = axes.flatten()

    results_summary = []

    for i, fname in enumerate(image_files):
        path = os.path.join(image_folder, fname)
        res = predict(model, path)
        results_summary.append(res)

        axes[i].imshow(res['image_pil'])
        color = '#2ecc71' if res['confidence'] >= 0.7 else '#e67e22' if res['confidence'] >= 0.4 else '#e74c3c'
        axes[i].set_title(f"{res['predicted_class']}\n{res['confidence']*100:.1f}%",
                          fontsize=10, fontweight='bold', color=color)
        axes[i].axis('off')

    # Hide unused axes
    for j in range(i + 1, len(axes)):
        axes[j].axis('off')

    plt.suptitle(f"Batch Inference — {n} images from '{image_folder}'",
                 fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.show()

    # Summary table
    print(f"\n{'#':<4} {'File':<30} {'Prediction':<15} {'Confidence':>10}")
    print("=" * 62)
    for i, (fname, res) in enumerate(zip(image_files, results_summary), 1):
        print(f"{i:<4} {fname:<30} {res['predicted_class']:<15} {res['confidence']*100:>9.2f}%")

    return results_summary

# ⬇️ Folder path ထည့်ပြီး run ပါ
# batch_results = batch_inference(model, "test_images_folder/")
print("ℹ️  batch_inference() ကို uncomment လုပ်ပြီး image folder path ထည့်ပါ။")

## 📝 Notes

- `IMAGE_PATH` ကို သင့် test image path နဲ့ ပြောင်းပါ
- `MODEL_PATH` ကို training notebook ကနေ save ထားတဲ့ `best_cnn_cifar10.pth` path ပြောင်းပါ
- GradCAM က `block3` (last conv block) ကို target layer အဖြစ်သုံးထားပါတယ်
- Confidence ≥70% = 🟢 Green | 40–70% = 🟡 Orange | <40% = 🔴 Red